## Movie Recommendation System - Data Cleaning

### Objective
This notebook cleans and prepares the MovieLens dataset for building a content-based recommendation system.

### Datasets
- `movies.csv`: movie metadata
- `ratings.csv`: user ratings
- `tags.csv`: user-generated tags

### Goal
Produce clean datasets:
- `movies_clean.csv`
- `ratings_clean.csv`
- `tags_clean.csv`

In [1]:
# IMPORT LIBRARIES
import pandas as pd
import numpy as np
import os

### Load Raw Data

In [2]:
path = "../data/raw/"

movies = pd.read_csv(f"{path}movies.csv")
ratings = pd.read_csv(f"{path}ratings.csv")
tags = pd.read_csv(f"{path}tags.csv")

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)

Movies: (9742, 3)
Ratings: (100836, 4)
Tags: (3683, 4)


### Inspect Movies Dataset

In [3]:
print("Preview of Dataset")
print(movies.head(2))

print("\nDataset Structure")
print(movies.info())

print("\nCheck for Missing Values in Each Column")
print(movies.isnull().sum())

Preview of Dataset
   movieId             title                                       genres
0        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy
1        2    Jumanji (1995)                   Adventure|Children|Fantasy

Dataset Structure
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 228.5 KB
None

Check for Missing Values in Each Column
movieId    0
title      0
genres     0
dtype: int64


### Clean Movie Dataset

Steps:
- Remove empty/null rows
- Extract year from title
- Clean title text
- Process genres

In [4]:
# REMOVE ROWS WHERE TITLE OR GENRES ARE MISSING
movies = movies.dropna(subset=["title", "genres"])

# EXTRACT YEAR
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)")

# CLEAN TITLE (remove year)
movies["title"] = movies["title"].str.replace(r"\(\d{4}\)", "", regex=True).str.strip()

# SPLITE GENRES INTO LIST
movies["genres"] = movies["genres"].str.replace("|", " ", regex=False)

### Inspect Ratings Dataset

In [5]:
print("Preview of Dataset")
print(ratings.head(2))

print("\nDataset Structure")
print(ratings.info())

print("\nCheck for Missing Values in Each Column")
print(ratings.isnull().sum())

Preview of Dataset
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247

Dataset Structure
<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB
None

Check for Missing Values in Each Column
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


### Clean Ratings Dataset

Steps:
- Convert timestamp to datetime
- Remove duplicates
- Validate rating values

In [6]:
# CONVERT TIMESTAMP
ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit='s')

# REMOVE DUPLICATES (keep latest)
ratings = ratings.sort_values("timestamp").drop_duplicates(
    subset=["userId", "movieId"], keep="last"
)

# KEEP VALID RATINGS ONLY
ratings = ratings[(ratings["rating"] >= 0.5) & (ratings["rating"] <= 5.0)]

ratings.head()

,userId,movieId,rating,timestamp
66719,429,595,5.0,1996-03-29 18:36:55
66716,429,588,5.0,1996-03-29 18:36:55
66717,429,590,5.0,1996-03-29 18:36:55
66718,429,592,5.0,1996-03-29 18:36:55
66712,429,432,3.0,1996-03-29 18:36:55


### Inspect Tags Dataset

In [7]:
print("Preview of Dataset")
print(tags.head(2))

print("\nDataset Structure")
print(tags.info())

print("\nCheck for Missing Values in Each Column")
print(tags.isnull().sum())

Preview of Dataset
   userId  movieId              tag   timestamp
0       2    60756            funny  1445714994
1       2    60756  Highly quotable  1445714996

Dataset Structure
<class 'pandas.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   userId     3683 non-null   int64
 1   movieId    3683 non-null   int64
 2   tag        3683 non-null   str  
 3   timestamp  3683 non-null   int64
dtypes: int64(3), str(1)
memory usage: 115.2 KB
None

Check for Missing Values in Each Column
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64


### Clean Tags Dataset

Steps:
- Convert to lowercase
- Remove duplicates
- Convert timestamp

In [8]:
# Convert timestamp
tags["timestamp"] = pd.to_datetime(tags["timestamp"], unit='s')

# Lowercase tags
tags["tag"] = tags["tag"].str.lower()

# Remove duplicates
tags = tags.drop_duplicates(subset=["userId", "movieId", "tag"])

tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,2015-10-24 19:29:54
1,2,60756,highly quotable,2015-10-24 19:29:56
2,2,60756,will ferrell,2015-10-24 19:29:52
3,2,89774,boxing story,2015-10-24 19:33:27
4,2,89774,mma,2015-10-24 19:33:20


### Cross Dataset Consitency
Ensure that all movieIds in ratings and tags exist in movies dataset

In [9]:
valid_movie_ids = movies["movieId"]

ratings = ratings[ratings["movieId"].isin(valid_movie_ids)]
tags = tags[tags["movieId"].isin(valid_movie_ids)]

### Save Cleaned Datasets

In [10]:
cleaned_file_path = "../data/cleaned/"

os.makedirs(cleaned_file_path, exist_ok=True)

movies.to_csv(cleaned_file_path + "movies_clean.csv", index=False)
ratings.to_csv(cleaned_file_path + "ratings_clean.csv", index=False)
tags.to_csv(cleaned_file_path + "tags_clean.csv", index=False)

print("Cleaned files saved!")

Cleaned files saved!


## Summary

### Movies
- Removed null rows
- Extracted year
- Cleaned titles
- Processed genres

### Ratings
- Converted timestamps
- Removed duplicates
- Validated ratings

### Tags
- Normalized text
- Removed duplicates

### Consistency
- Ensured valid movieId references

---